# Assignment Hyperparameter Optimization

In [ ]:
!pip uninstall tf-keras
!pip install keras-tuner
!pip install tensorflow==2.16.1

Found existing installation: tf_keras 2.18.0
Uninstalling tf_keras-2.18.0:
  Would remove:
    /usr/local/lib/python3.11/dist-packages/tf_keras-2.18.0.dist-info/*
    /usr/local/lib/python3.11/dist-packages/tf_keras/*
Proceed (Y/n)? Y
  Successfully uninstalled tf_keras-2.18.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 589.8/589.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 110.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  A

In [ ]:
import keras
import tensorflow as tf
print("Keras Current Version:", keras.__version__, "Tensorflow Current Version:", tf.__version__)

Keras Current Version: 3.8.0 Tensorflow Current Version: 2.18.0


# Imports

In [ ]:
import numpy as np
import pandas as pd
from joblib import dump, load
import random
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.initializers import RandomNormal, RandomUniform, GlorotUniform, GlorotNormal, HeNormal
from keras.optimizers.schedules import ExponentialDecay
from keras_tuner import RandomSearch, GridSearch, BayesianOptimization
from keras_tuner.engine.hyperparameters import HyperParameters

random.seed(46)
np.random.seed(46)
tf.random.set_seed(46)

# import os
import time


# Functions

In [ ]:
def preprocess_data(filepath):
    data = pd.read_csv(filepath)
    scaler = StandardScaler()
    X = scaler.fit_transform(data.drop('Outcome', axis=1))
    y = data['Outcome'].values
    dump(scaler, 'scaler.joblib')
    return X, y

def prepare_datasets(X_train, X_val, y_train, y_val, batch_size=None):
    if batch_size is None:
        batch_size = len(X_train)
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.shuffle(buffer_size=len(X_train)).batch(batch_size)
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_dataset = val_dataset.batch(batch_size)
    return train_dataset, val_dataset

def plot_training_history(history, train_loss='loss', train_metric='accuracy', val_loss='val_loss', val_metric='val_accuracy'):

    #Loss
    plt.figure(figsize=(10, 5))
    plt.plot(history.history[train_loss], label='Training Loss')
    plt.plot(history.history[val_loss], label='Validation Loss')
    plt.title('Training and Validation Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    # Metrics
    plt.figure(figsize=(10, 5))
    plt.plot(history.history[train_metric], label=f"Training: {train_metric}")
    plt.plot(history.history[val_metric], label=f"Validation: {val_metric}")
    plt.title(f'Training and Validation {train_metric} Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel(f'train_metric')
    plt.legend()
    plt.show()

def get_best_epoch_details(history):
    val_losses = history.history['val_loss']
    min_val_loss_index = val_losses.index(min(val_losses))
    best_epoch = min_val_loss_index + 1

    epoch_details = {}
    for key in history.history.keys():
        epoch_details[key] = history.history[key][min_val_loss_index]

    epoch_details['best_epoch'] = best_epoch
    print(f"Best epoch details: {epoch_details}")

# Data Preparation

In [ ]:
X, y = preprocess_data('/content/diabetes.csv')

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

train_ds, val_ds = prepare_datasets(X_train, X_val, y_train, y_val, batch_size=32)

# Task 1: Hiperparametre arama uzayını aşağıdaki değerlere göre oluşturunuz:

**Layer sayısı**:  1-10

**Unit sayısı**: 32-512 arasında 16'şar artacak şekilde.

**Aktivasyon fonksiyonları**: relu, tanh, sigmoid

**l2**: 0.0001-0.01

**dropout**: 0.1-0.5 arasında 0.05 artacak şekilde.

**initial learning rate**: 0.0001-0.01 (1e-4 - 1e-2)

**learning rate scheduler**: decay steps: 20

**optimizer'lar**: 'sgd', 'adam', 'rmsprop' (olduğu gibi kalabilir)

**Random search:** epoch sayısı en az 200 olmalı

Diğer ayarları dilediğiniz gibi yapabilirsiniz.

In [ ]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(train_ds.element_spec[0].shape[1],)))

    # Hidden layers, activation functions, l2, Dropout
    for i in range(hp.Int('num_layers',1,10)):

        model.add(Dense(units=hp.Int('units_'+str(i),min_value=32,max_value=512,step=16),
                        activation=hp.Choice('activation_' + str(i), values= ['relu','tanh','sigmoid']),
                        kernel_regularizer=l2(hp.Float('kernel_regularizer',min_value=0.0001,max_value=0.01))))

        model.add(BatchNormalization())
        model.add(Dropout(hp.Float('dropout',min_value=0.1,max_value=0.5,step=0.05)))

    model.add(Dense(1, activation='sigmoid'))

    # Learning rate schedule
    initial_learning_rate = hp.Float('learning_rate',min_value=0.0001,max_value=0.01)

    lr_schedule = ExponentialDecay(
        initial_learning_rate=initial_learning_rate,
        decay_steps=20,
        decay_rate=0.96,
        staircase=True
    )

    # optimizers
    optimizer_choice = hp.Choice('optimizer', values=['sgd', 'adam', "rmsprop"])
    if optimizer_choice == 'sgd':
        optimizer = SGD(
            learning_rate=lr_schedule,
            momentum=hp.Float('momentum', min_value=0.0, max_value=0.9, step=0.1)
        )
    elif optimizer_choice == 'adam':
        optimizer = Adam(
            learning_rate=lr_schedule,
            beta_1=hp.Float('beta1', min_value=0.85, max_value=0.99, step=0.01),
            beta_2=hp.Float('beta2', min_value=0.999, max_value=0.9999, step=0.0001),
            epsilon=hp.Float('epsilon', min_value=1e-8, max_value=1e-7, step=1e-8)
        )

    elif optimizer_choice == 'rmsprop':
        optimizer = RMSprop(
            learning_rate=lr_schedule,
            rho=hp.Float('rho', min_value=0.8, max_value=0.99, step=0.01),
            epsilon=hp.Float('epsilon', min_value=1e-10, max_value=1e-8, step=1e-10),
            momentum=hp.Float('momentum', min_value=0.0, max_value=0.9, step=0.1)
        )

    model.compile(optimizer=optimizer,
                  loss="binary_crossentropy",
                  metrics=["accuracy"])

    return model

# Task 2: Epoch sayısı 200 olacak şekilde aramayı başlatınız. Diğer ayarlar aynı kalabilir.



In [ ]:
random_search_tuner = RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=20,
    executions_per_trial=1,
    overwrite=True
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=20,
    verbose=1,
    restore_best_weights=True
)

In [ ]:
random_search_tuner.search(train_ds,
                           epochs=200,
                           validation_data=val_ds,
                           callbacks=[early_stopping])

Trial 20 Complete [00h 00m 42s]
val_loss: 0.5086183547973633

Best val_loss So Far: 0.5074795484542847
Total elapsed time: 00h 11m 57s


In [ ]:
random_search_tuner.search_space_summary()

Search space summary
Default search space size: 30
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 10, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 16, 'sampling': 'linear'}
activation_0 (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'tanh', 'sigmoid'], 'ordered': False}
kernel_regularizer (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'linear'}
dropout (Float)
{'default': 0.1, 'conditions': [], 'min_value': 0.1, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'linear'}
optimizer (Choice)
{'default': 'sgd', 'conditions': [], 'values': ['sgd', 'adam', 'rmsprop'], 'ordered': False}
momentum (Float)
{'default': 0.0, 'conditions': [], 'min_value': 0.0, 'max_value':

# Task 3: En iyi 3 hiperparametre setini getiriniz, ayrı ayrı kaydediniz, değerlerini inceleyiniz, bazı hiperparametre değerlerini yorumlayınız.

In [ ]:
best_hps = random_search_tuner.get_best_hyperparameters(num_trials=3)

In [ ]:
best_hps_1 = best_hps[0]
best_hps_2 = best_hps[1]
best_hps_3 = best_hps[2]

In [ ]:
print(f"Best hyperparameters: {best_hps_1.values}")

Best hyperparameters: {'num_layers': 7, 'units_0': 448, 'activation_0': 'tanh', 'kernel_regularizer': 0.0077508023452509825, 'dropout': 0.1, 'learning_rate': 0.00778115296551228, 'optimizer': 'rmsprop', 'momentum': 0.30000000000000004, 'units_1': 320, 'activation_1': 'relu', 'units_2': 208, 'activation_2': 'tanh', 'units_3': 208, 'activation_3': 'tanh', 'units_4': 48, 'activation_4': 'tanh', 'beta1': 0.88, 'beta2': 0.9999, 'epsilon': 3.0000000000000004e-08, 'rho': 0.9, 'units_5': 272, 'activation_5': 'sigmoid', 'units_6': 288, 'activation_6': 'relu', 'units_7': 160, 'activation_7': 'sigmoid', 'units_8': 32, 'activation_8': 'sigmoid', 'units_9': 32, 'activation_9': 'tanh'}


In [ ]:
print(f"Best hyperparameters: {best_hps_2.values}")

Best hyperparameters: {'num_layers': 7, 'units_0': 48, 'activation_0': 'tanh', 'kernel_regularizer': 0.008373587238413216, 'dropout': 0.25, 'learning_rate': 0.0034363664449084926, 'optimizer': 'rmsprop', 'momentum': 0.8, 'units_1': 320, 'activation_1': 'sigmoid', 'units_2': 208, 'activation_2': 'relu', 'units_3': 176, 'activation_3': 'relu', 'units_4': 144, 'activation_4': 'tanh', 'beta1': 0.94, 'beta2': 0.9998, 'epsilon': 8e-08, 'rho': 0.9700000000000001, 'units_5': 480, 'activation_5': 'relu', 'units_6': 192, 'activation_6': 'tanh', 'units_7': 192, 'activation_7': 'tanh', 'units_8': 96, 'activation_8': 'tanh', 'units_9': 192, 'activation_9': 'sigmoid'}


In [ ]:
print(f"Best hyperparameters: {best_hps_3.values}")

Best hyperparameters: {'num_layers': 2, 'units_0': 224, 'activation_0': 'tanh', 'kernel_regularizer': 0.00785086823637372, 'dropout': 0.5, 'learning_rate': 0.007950235153936525, 'optimizer': 'rmsprop', 'momentum': 0.1, 'units_1': 432, 'activation_1': 'tanh', 'units_2': 400, 'activation_2': 'relu', 'units_3': 272, 'activation_3': 'sigmoid', 'units_4': 464, 'activation_4': 'tanh', 'beta1': 0.97, 'beta2': 0.9991, 'epsilon': 2e-08, 'rho': 0.93, 'units_5': 480, 'activation_5': 'tanh', 'units_6': 240, 'activation_6': 'relu', 'units_7': 272, 'activation_7': 'tanh', 'units_8': 160, 'activation_8': 'relu', 'units_9': 224, 'activation_9': 'relu'}


# Task 4: En iyi 3 modeli seçiniz.

In [ ]:
best_models = random_search_tuner.get_best_models(num_models=3)

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 1 variables whereas the saved optimizer has 61 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 1 variables whereas the saved optimizer has 21 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


# Task 5: En iyi 3 modelin bir döngü aracılığı ile model başarısını hesaplayınız

In [ ]:
for i, model in enumerate(best_models):
    loss, acc = model.evaluate(val_ds, verbose=0)
    print(f"Model {i+1}, Validation loss: {loss}, Validation Accuracy: {acc}")

Model 1, Validation loss: 0.5074795484542847, Validation Accuracy: 0.7857142686843872
Model 2, Validation loss: 0.5086183547973633, Validation Accuracy: 0.798701286315918
Model 3, Validation loss: 0.5110181570053101, Validation Accuracy: 0.7597402334213257


# Task 6: Modellerin accuracy değerleri arasında neden fark var? En tepedeki modelin en iyi olmasını bekleriz, eğer öyle değilse neden en tepedekinin accuracy değeri en yüksek değil?

Her seferinde farklı accuracy değerler almamızın sebebi unit sayısı , dropout değeri,learning rate her seferinde farkl; olarak seçilmiş.Yani farklı kombinasyonlardan farklı accuracy değerleri almış oluyoruz.

Accuracy değerleri arasındakı fark loss -la ilgili olmalı. 2 ci modelde validation loss daha yüksek ve bizim önceleğimiz val_loss sanırım o yüzden.